# Patent analysis

In [1]:
import pandas as pd

In [2]:
# AltairSaver = altair_save_utils.AltairSaver()

In [3]:
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import utils
import importlib
importlib.reload(utils);

2024-06-04 15:48:51,448 - botocore.credentials - INFO - Found credentials in environment variables.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-06-04 15:48:52,918 - datasets - INFO - PyTorch version 2.1.2 available.


## Load data

In [4]:
# Labelled data
data_df = utils.load_patents_data().query("topics != 'arts'").query("country_code != 'CN'")

In [5]:
# Taxonomy dataframe
topics_df = utils.load_topic_data()

In [6]:
# Transform to one id and topic pair per row
data_exploded_df = utils.explode_data(data_df).query("topic != 'arts'")

## Baseline trends

Baseline trends for patent counts

In [7]:
importlib.reload(utils);
baseline_df = utils.get_baseline_patents()

In [8]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = baseline_df,
    year_start = 2019,
    year_end = 2023  
)
trends_baseline

,magnitude,growth
counts,7572871.2,31.192759


In [9]:
fig = pu.ts_smooth(
    baseline_df.assign(Total="Total").assign(counts = lambda df: df.counts/1e+6),
    ["Total"],
    variable= "counts",
    variable_title = "Publications (millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

## Insight 0: Overall trends

Early-years project growth of funding and project counts trends



In [14]:
ts_counts = utils.get_timeseries(data_df, column='id')

In [15]:
ts_counts

,year,counts
0,2013,345
1,2014,552
2,2015,767
3,2016,851
4,2017,833
5,2018,817
6,2019,803
7,2020,785
8,2021,859
9,2022,801


In [16]:
au.ts_magnitude_growth_(
    ts_df = ts_counts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,790.6,-3.587444


In [17]:
fig = pu.ts_smooth(
    ts_counts.assign(Total="Total"),
    ["Total"],
    variable= "counts",
    variable_title = "Publications",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

In [28]:
utils.get_data_distribution(data_exploded_df.query("year >= 2019"), column='type', values=['id'])

,type,counts,counts_prop
0,Biosciences,159,0.04
1,Child care & preschool,78,0.02
2,Development & learning,239,0.06
3,General,3009,0.761
4,Health,977,0.247
5,Parenting,16,0.004
6,Social,25,0.006
7,Technology,945,0.239


In [19]:
importlib.reload(utils);
ts_df = (
    utils.get_data_distribution(data_exploded_df, column='type', values=['id'], ts=True)
    .query("type != 'General'")
)
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='type', value='id')

,magnitude,growth,type,counts
7,3.2,200.000000,Parenting,16
5,5.0,125.000000,Social,25
2,47.8,37.719298,Development & learning,239
6,189.0,20.000000,Technology,945
4,195.4,-1.500000,Health,977
0,31.8,-2.083333,Biosciences,159
1,15.6,-2.083333,Child care & preschool,78
3,601.8,-6.694344,General,3009


In [20]:
fig = pu.ts_smooth(
    ts_df,
    ts_df['type'].unique(),
    variable= "counts",
    variable_title = "",
    category_column = 'type',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 1: Technology trends

- Magnitude and growth for technology topic overall
- Distribution of different technologies
- Growth of different technologies in UKRI funding


### Overall technology topic growth

In [21]:
tech_subtypes = set(topics_df.query("type == 'Technology'").subtype.unique())
tech_subtypes

{'AI', 'Immersive tech', 'Internet', 'Mobile'}

In [22]:
tech_type_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'type'])
)

In [23]:
# ts_amounts_tech = utils.get_timeseries(tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(tech_type_df, column='id')
utils.plot_quick_ts(ts_counts_tech, 'counts')

alt.Chart(...)

In [24]:
au.ts_magnitude_growth_(ts_counts_tech, year_start = 2019, year_end = 2023)

,magnitude,growth
counts,189.0,20.0


### Distribution of different technologies

In [29]:
tech_subtype_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    .query("type == 'Technology'")
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'subtype'])
)

In [30]:
# Total tech funding
counts_total = tech_subtype_df.drop_duplicates('id').query("year >= 2019").id.nunique()

In [31]:
tech_subtype_dist = (
    tech_subtype_df
    .query("year >= 2019")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
    .assign(counts_prop = lambda df: df.counts/counts_total)
)

tech_subtype_dist

,subtype,counts,counts_prop
0,AI,640,0.677249
1,Immersive tech,300,0.317460
2,Internet,3,0.003175
3,Mobile,229,0.242328


### Growth of technology topics

In [32]:
column = 'subtype'
value = 'counts'

tech_subtype_ts = (
    tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

utils.magnitude_and_growth(tech_subtype_ts, column, value)

,magnitude,growth,subtype
0,128.0,24.203822,AI
0,60.0,45.384615,Immersive tech
0,0.6,-50.000000,Internet
0,45.8,-16.339869,Mobile


In [95]:
fig = pu.ts_smooth(
    tech_subtype_ts,
    ["AI", "Immersive tech", "Internet", "Mobile"],
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 2: Applications

- Where are these technologies applied the most?
- Where do we see growth vs stagnation when it comes to applications?

In [36]:
tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

tech_ids_5y = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2019")
    .drop_duplicates('id')
    .id.to_list()
)

### Application distribution

In [42]:
column = 'type'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology' and type != 'General'"),
    column=column, 
    values=['id'],
    ts=True
)



In [43]:
tech_applications_df

,type,counts,counts_prop
0,Biosciences,36,0.038
1,Child care & preschool,11,0.012
2,Development & learning,48,0.051
3,General,756,0.8
4,Health,239,0.253
5,Parenting,4,0.004
6,Social,2,0.002
7,Technology,945,1.0


In [44]:
fig = pu.ts_smooth(
    tech_applications_ts,
    tech_applications_ts[column].unique(),
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [45]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')

,magnitude,growth,type,counts
7,0.4,inf,Social,2
5,7.2,130.000000,Biosciences,36
6,0.8,100.000000,Parenting,4
1,9.6,84.210526,Development & learning,48
4,189.0,20.000000,Technology,945
2,151.2,14.179104,General,756
3,47.8,2.238806,Health,239
0,2.2,-50.000000,Child care & preschool,11


In [41]:
pd.set_option('display.max_colwidth', 200)
(
    data_exploded_df
    .query('id in @tech_ids')
    # .query("subtype == 'Personal social emotional'")
    .query("type == 'Social'")
    .drop_duplicates(['id'])
    .sort_values('year', ascending=False)
)[['id', 'text', 'topics', 'year']]

,id,text,topics,year
2846,KR-20210111483-A,Happy communication support system in connection with kindergarten and nursery school. The present invention is a communication web/app for parents and teachers of kindergartens and daycare center...,social_services,2021
4112,KR-102267147-B1,"A system which manages the use time of child-care institution. The present invention relates to a childcare institution use time management system, and in the present invention, the &lt;informatio...",social_services,2021


### Application distribution: More granular subtypes

In [116]:
column = 'subtype'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id'],
    ts=True
)
tech_applications_df.query("type != 'Technology'").sort_values('counts', ascending=False)

,subtype,counts,counts_prop,type
8,Infancy,1501,0.77,General
23,Sleep,335,0.172,Health
20,Physical development,147,0.075,Health
6,Health,102,0.052,Health
4,Games,80,0.041,General
14,Neuroscience,68,0.035,Biosciences
22,Preschool,66,0.034,Child care & preschool
21,Prenatal,38,0.019,Health
25,Special educational needs,34,0.017,Development & learning
15,Non-tech assessments,32,0.016,General


In [118]:
(
    utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')
    # .sort_values(['type', 'growth'], ascending=False)
    .sort_values('growth', ascending=False)
)

,magnitude,growth,subtype,counts,type
0,0.4,inf,Social services,2,Social
1,5.8,1200.000000,Communication and language,29,Development & learning
2,2.4,400.000000,Cognitive development,12,Development & learning
3,6.4,271.428571,Non-tech assessments,32,General
4,13.6,200.000000,Neuroscience,68,Biosciences
5,2.4,125.000000,Nutrition & weight,12,Health
6,7.6,78.571429,Prenatal,38,Health
7,6.8,66.666667,Special educational needs,34,Development & learning
8,3.4,66.666667,Literacy,17,Development & learning
9,1.0,50.000000,Parenting,5,Parenting


In [105]:
cat_type = 'Development & learning'
cats = list(topics_df.query("type == @cat_type").subtype.unique())

In [106]:
fig = pu.ts_smooth(
    tech_applications_ts,
    cats,
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

### Which technology is applied to most to subtype X?

## Insight 3: Geographical insights

- Top countries in terms of counts
- UK vs baseline growth for overall counts, in technology counts and application counts

In [205]:
# data_exploded_df.explode('country_code').drop_duplicates(['id', 'country_code']).isnull().sum()
# Consider lack of information

In [108]:
data_countries_df = (
    data_exploded_df
    # .explode('country_code')
    .dropna(subset=['country_code'])
    .query("type == 'Technology'")
    .query('subtype in @tech_subtypes')
    .drop_duplicates(['id'])    
)
country_codes = data_countries_df.country_code.unique()

growth_df = []
ts_counts = []
for country_code in country_codes:
    country_df = data_countries_df.query("country_code == @country_code")
    _ts_counts = utils.get_timeseries(country_df, column='id').assign(country_code = country_code)
    growth_df.append(
        au.ts_magnitude_growth_(
            ts_df = _ts_counts,
            year_start = 2019,
            year_end = 2023  
        )
        .assign(country_code = country_code)
        .reset_index(drop=True)
    )
    ts_counts.append(_ts_counts)
growth_df = pd.concat(growth_df, ignore_index=True)
ts_counts = pd.concat(ts_counts, ignore_index=True)

In [111]:
(
    growth_df
    .sort_values('magnitude', ascending=False)
    .head(20)
)

,magnitude,growth,country_code
0,250.6,-8.729140,CN
5,53.0,44.166667,KR
3,30.4,-13.978495,US
4,16.4,-14.285714,WO
29,9.4,29.166667,JP
1,5.8,120.000000,EP
9,4.2,40.000000,TW
23,3.4,700.000000,AU
13,2.6,100.000000,TR
14,2.2,700.000000,DE


In [110]:
countries = ['US', 'GB', 'CN', 'KR', 'JP']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "counts",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

In [61]:
data_countries_df = (
    data_exploded_df
    .explode('country_code')
    .dropna(subset=['country_code'])
    .query("type == 'Technology'")
    .query('subtype in @tech_subtypes')
    .drop_duplicates(['id', 'subtype']) 
    .query("country_code == 'GB'")   
)

In [62]:
data_countries_df.groupby('subtype').agg(counts=('id', 'nunique')).reset_index()

,subtype,counts
0,AI,9
1,Immersive tech,2
2,Mobile,4
